<!-- # Intro To Agents Part 1: Tools, MCPs & Tracing
--------------------

1. Introduction

2. Agents & Tools with LangChain

3. Model Context Protocols (MCPs) with FastMCP

4. Agents & Tracing with LangSmith

5. Next Steps -->

## Agents, Tools, MCPs and All That
---------

## 1. Introduction

It's been a while since I posted. It's not as easy to write anymore with my family commitments, but its actually never been easier to do work with LLMs and AI more generally! Definitely the biggest developments in the last year or two have been on Agents and Agentic AI applications. To be honesy, I've always been a little skeptical about Agents and AI more generally. In science, they train you to be skeptical, as Feynman said, ["The first principle is that you must not fool yourself—and you are the easiest person to fool."](http://brainyquote.com/quotes/richard_p_feynman_137642) However, over the last two years I've very bought into AI and have switched roles to become an AI Engineer!

In this post I want to talk about Agents, Tools, MCPs and All That (the title being a play on the famous Vector Calculus book [Div, Grad, Curl and All that](https://www.google.com/books/edition/Div_Grad_Curl_and_All_that/sembQgAACAAJ?hl=en) that I read in undergrad) which are the newest crazes in AI and technology more broadly. I'll keep this post brief and simple. Partly because long posts are harder to write to, but also because people dont have attention anymore!

I'll go over how to buid a simple agent, use a [MCP](https://en.wikipedia.org/wiki/Model_Context_Protocol) server and observe agent behavoir; all using [LangChain](https://www.langchain.com/), [Groq](https://groq.com/), [FastMCP](https://gofastmcp.com/getting-started/welcome) and [LangSmith](https://www.langchain.com/langsmith-platform). The agent will be a simple ReAct agent. It will have tools that can help us find weather (like everyones first agent), but also help find the closest Police station and public restroom in NYC (data coming from [OpenData NYC](https://data.cityofnewyork.us/)). Very helpful things! In the back end, I'll use [MongoDB](https://www.mongodb.com/), [Redis](https://redis.io/) along with a handful of APIs to accompish these tasks.

Let's get into it, I'll first start with a bunch of import and then get into what an agent is:


In [15]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent

## 1. Agents

Large language models (LLMs) have been round for a few years at this point and most people are familiar with them through chatbots such as ChatGPT. With the advent of ChatGPT, people saw a AI model could have converations and write prose that was similar to a humans ability. LLMs were trained on an enormous amount of text data from the internet and can answer almost any question. I still remember in late 2022 asking ChatGPT to prove the uniqueness of solutions to the diffusion equation, it crushed the answer in 2 seconds. I asked more questiosn from semiconductor physics, quantum mechanics and numerical analysis, it got them all correct. I was impressed, but also realized it was reguritating things from the internet. Here's an example of a simple LLM with [LangChain](https://www.langchain.com/)

In [20]:
model = ChatGroq(model="openai/gpt-oss-120b")
llm  = model | StrOutputParser() 

print(llm.invoke("Give me a 100 word explanation of what a Sobolev Space is."))

A Sobolev space is a mathematical construct that extends the concept of differentiability to functions whose derivatives may exist only in an averaged, weak sense. Formally, for an integer k and real p ≥ 1, the Sobolev space W^{k,p}(Ω) consists of functions on a domain Ω whose weak derivatives up to order k are integrable to the p‑th power. These spaces provide a natural setting for studying partial differential equations, variational problems, and approximation theory, because they combine normed structure with flexibility, allowing limits of smooth functions to remain within the space. Consequently, Sobolev spaces are essential tools in modern analysis, linking functional analysis with geometric and physical applications.


Amazing right? 

Now ask it something simple:

In [12]:
print(llm.invoke("What happened on July 4th 2026?"))

I’m sorry, but I don’t have any information about events that occurred on July 4 2026. My training data only goes up through June 2024, so I can’t provide details about anything that happened after that date. If you have a more recent source you’d like to discuss, feel free to share it!


While the model has been able to compress so much of humanity's knowledge into 120 billion parameters; it doesnt know something that happened after it was trained. An LLM deployed to answer questions for employees also doesnt know about things specific to your company.

Engineers solved this by using [Retrivial Augument Generation](https://michael-harmon.com/posts/rag_jfk2/), but this requires you to keep an up-to-date knowledge-base. People started introducing tools that allow the LLM to take actions (like search the web). Now the LLM can look up things it doesn't know, but it also allows it to take actions on your behalf and interact with its environment (like edit files, sumamrize emails, send messages, etc.). An agent is an LLM with the a set of tools. The simplest agent reasons about what steps to take based on a reques and its itnernal state. It can uses tools to take actions or generate text based on the internal state and the context (text and information you have provided as well as its collected). This is called a [ReAct agent](https://arxiv.org/abs/2210.03629).

The ability of a LLM to reason and also to take actions with tools has lead to a revolution in technology. We'll go over the basics of tools next.

## 2. Agents & Tools

Let's go over first the tool everyone stars with, which is the ability to get the weather. A tool is a regular function; however it needs to be wrapped in a decorator to declare it one. We'll go over that in a second, but for now I'll import function:

In [21]:
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)
from mymcp.server import get_weather

The defintion of this function is [here](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py). It is a simple function that takes in a city and queries the [OpenWeather API](https://openweathermap.org/) to give us the current weather:

In [22]:
get_weather("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 25.12,
  'feels_like': 25.2,
  'temp_min': 23.21,
  'temp_max': 26.19,
  'pressure': 1011,
  'humidity': 58,
  'sea_level': 1011,
  'grnd_level': 1010},
 'visibility': 10000,
 'wind': {'speed': 5.66, 'deg': 250},
 'clouds': {'all': 0},
 'dt': 1787532946,
 'sys': {'type': 1,
  'id': 4610,
  'country': 'US',
  'sunrise': 1787480061,
  'sunset': 1787528603},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

How does an LLM call this function? That's where tools come in. They are decorators around the Python function that give the LLM enough information on when and how to use the function. The function annotations and the doc string are passed into the context window of the LLM so that it knows what its working with. 

We can create a weather tool for the LLM to use by importing from LangChain and then explictly make a tool called "`get_weather`" with the following (the decorator method with [FastMCP](http://gofastmcp.com/getting-started/welcome) is [here](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py)):

In [26]:
from langchain.tools import tool
weather_tool = tool("get_weather", get_weather)

The tool is actually now a co-routine that can be called with the async-invoke method:

In [27]:
await weather_tool.ainvoke("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 25.12,
  'feels_like': 25.2,
  'temp_min': 23.21,
  'temp_max': 26.19,
  'pressure': 1011,
  'humidity': 58,
  'sea_level': 1011,
  'grnd_level': 1010},
 'visibility': 10000,
 'wind': {'speed': 5.66, 'deg': 250},
 'clouds': {'all': 0},
 'dt': 1787532946,
 'sys': {'type': 1,
  'id': 4610,
  'country': 'US',
  'sunrise': 1787480061,
  'sunset': 1787528603},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

Now we can use the [create_agent](https://reference.langchain.com/python/langchain/agents/factory/create_agent) to create a simple ReAct agent:

In [28]:
agents = create_agent(model=model, tools=[weather_tool])

Now I can pass the query in and use the agent using the same asynchronous `ainvoke` method,

In [29]:
query = "What is the weather in New York?"

In [30]:
result = await agents.ainvoke({'messages': [{'role': 'user', 'content': query}]})
messages = result.get("messages")


The ReAct agent returns a list of the all messages in the conversation:

In [31]:
messages

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='9aa90351-2efe-4446-8350-f0b87e5d88c8'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is the weather in New York?" Need to call get_weather function with city "New York". Use function.', 'tool_calls': [{'id': 'fc_1908a057-388e-4232-82d2-28c608f0632d', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 261, 'total_tokens': 317, 'completion_time': 0.118889854, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.022129046, 'prompt_tokens_details': None, 'queue_time': 0.078422765, 'total_time': 0.1410189}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0adba2bb92', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0316d-2483-75

Notice the conversation has a `HumanMessage` (prompt) -> `AIMessage` where the agent reasons what to do. In this case it only has one tool and calls it which returns a `ToolMessage`. The `ToolMessage` is read by the AI agent and synthesizes its message.  

With a long chain of reasoning and actions viewing the contents as a list isnt' easy. Instead most people use tracing and observability tools like [LangSmith](https://www.langchain.com/langsmith/observability), which I'll go over later.

## 3. Model Context Protocols (MCPs)

A problem with tools is that they have to be localized and requires access to source code, data and other resources that developers might not wantt users to have access to. Additionally, each LLM and agent framework (OpenAI, Google, Anthropic, etc.) has their own way of defining and using tools. This can cause headaches for developers that need to make tools work across these various frameworks. The [**Model Context Protocol (MCP)**](https://en.wikipedia.org/wiki/Model_Context_Protocol) is an open standard for connecting LLM applications to external tools and data sources. Without MCP, each application must implement a custom integration for every tool. For example, a LangChain agent might need separate code for weather APIs, databases, file systems, and internal services.

MCP provides a common interface:

- **MCP server**: Exposes tools, resources, or prompts.
- **MCP client**: Connects an application or agent to the server.
- **Tool**: A callable operation, such as `get_weather`.
- **Schema**: Describes the tool's arguments and return values.

Model context protocol solves a few problems:

1. **Standardized integration**  
    Tools can be exposed through the same protocol instead of requiring custom integrations.

2. **Reusability**  
    One MCP server can be used by multiple agents and applications.

3. **Tool discovery**  
    Clients can list available tools and inspect their descriptions and input schemas.

4. **Separation of concerns**  
    The MCP server handles API calls and business logic, while the LLM application handles reasoning and conversation.

5. **Remote access**  
    Tools can run in another process or on another machine and be accessed over HTTP or other supported transports.

In some ways, MCP is for agents as the way APIs are for regular code. The standard MCP framework is [FastMCP](https://gofastmcp.com/getting-started/welcome). I created a FastMCP server here with several tools I'll discuss. 

In [ ]:
from fastmcp import Client

In [ ]:
mcp_client = Client("http://localhost:8080/mcp")

In [ ]:
async with mcp_client:
    tools = await mcp_client.list_tools()

In [ ]:
tools

In [ ]:
async with mcp_client:
    result = await mcp_client.call_tool("get_weather", {"city": "New York"})

In [ ]:
async with mcp_client:
    point = await mcp_client.call_tool("convert_address_to_point", {"address": "567 Ocean Avenue, Brooklyn, NY 11226"})

In [ ]:
from mymcp.server import find_closest_restroom

pt = json.loads(point.content[0].text)
find_closest_restroom(pt["lat"], pt["lng"])

In [ ]:
async with mcp_client:
    result = await mcp_client.call_tool("find_closest_restroom", {"lat": json.loads(point.content[0].text)["lat"], "lng": json.loads(point.content[0].text)["lng"]})

In [ ]:
async with mcp_client:
    result = await mcp_client.call_tool("get_police_precinct", {"lat": json.loads(point.content[0].text)["lat"], "lng": json.loads(point.content[0].text)["lng"]})

In [ ]:
result

## 4. Agents & MCPs

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": "http://localhost:8080/mcp", "transport": "http"}})

In [ ]:
tool_list = await lc_client.get_tools()

In [ ]:
tool_list

In [ ]:
agent = create_agent(model=model, tools=tool_list)

In [ ]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattan, manhattan in?'
}]})

In [ ]:
print(result.get("messages")[-1].content)

In [ ]:
# result = await agent.ainvoke({
#     'messages': [{'role': 'user', 
#                  'content': 'Where is the closet bathroom to 625 Atlantic Ave, Brooklyn?'
# }]})
# print(result.get("messages")[-1].content)

In [ ]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'Whats the temparture in Boston?'
}]})
print(result.get("messages")[-1].content)

<!--  -->

## 5. Agent Observability With Langsmith

## 6. Next Steps